# Calculate Cell Features

Load images and localization results for feature extraction:

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import random
import pyvips
import numpy as np
import cv2
from tqdm import tqdm
import pandas as pd
from scripts.filters import *
from scripts.utils import *
from cell_analysis.extract_features import *

IMAGE_DIR = "/mnt/d/microglia_data"
imgs = [f for f in Path(IMAGE_DIR).glob("./*.tiff")]
PARQUET_DIR = "../cell_localization/clustered_bboxes"
parquet = [f for f in Path(PARQUET_DIR).glob("./*.parquet")]
classes = list(map((lambda x: 0 if "EV" in x else 1), [f.name for f in imgs]))

indices = [0, 9, 2, 3, 4, 11, 10, 7, 8, 5, 6, 1]
parquets = [parquet[i] for i in indices]

print(imgs)
print(parquets)
print(classes)

[PosixPath('/mnt/d/microglia_data/TPO_60.tiff'), PosixPath('/mnt/d/microglia_data/TPO_61_EV.tiff'), PosixPath('/mnt/d/microglia_data/TPO_62_EV2.tiff'), PosixPath('/mnt/d/microglia_data/TPO_62_EV.tiff'), PosixPath('/mnt/d/microglia_data/TPO_63_TPO.tiff'), PosixPath('/mnt/d/microglia_data/TPO_64_EV2.tiff'), PosixPath('/mnt/d/microglia_data/TPO_64_EV.tiff'), PosixPath('/mnt/d/microglia_data/TPO_65_TPO2.tiff'), PosixPath('/mnt/d/microglia_data/TPO_65_TPO.tiff'), PosixPath('/mnt/d/microglia_data/TPO_66_EV2.tiff'), PosixPath('/mnt/d/microglia_data/TPO_66_EV.tiff'), PosixPath('/mnt/d/microglia_data/TPO_67_TPO.tiff')]
[PosixPath('../cell_localization/clustered_bboxes/TPO_60_bboxes.parquet'), PosixPath('../cell_localization/clustered_bboxes/TPO_61_EV_bboxes.parquet'), PosixPath('../cell_localization/clustered_bboxes/TPO_62_EV2_bboxes.parquet'), PosixPath('../cell_localization/clustered_bboxes/TPO_62_EV_bboxes.parquet'), PosixPath('../cell_localization/clustered_bboxes/TPO_63_TPO_bboxes.parquet'

Calculate cell features and save to combined file.

In [ ]:
from tqdm import tqdm
import random

random.seed(25)
skips = 0
records = []

for idx, file in enumerate(parquets):
    bbs = extract_bbs_centroid(str(file), [])
    vips_img = pyvips.Image.new_from_file(imgs[idx], access="sequential")

    for bb in tqdm(bbs):

        x1, y1, x2, y2, cx, cy = (
            bb["x1"],
            bb["y1"],
            bb["x2"],
            bb["y2"],
            bb["cx"],
            bb["cy"],
        )
        crop = vips_img.crop(x1, y1, x2 - x1, y2 - y1)
        patch = np.ndarray(
            buffer=crop.write_to_memory(),
            dtype=np.uint8,
            shape=[crop.height, crop.width, crop.bands],
        )

        if patch.shape[-1] > 3:
            patch = patch[:, :, :3]
        try:
            feats = calculate_features(patch, (cx, cy))
        except Exception:
            skips += 1
            print(skips)
            continue
        if feats is None:
            skips += 1
            print(skips)
            continue
        else:
            (
                soma_area,
                cell_area,
                tortuosity,
                terminal_pts,
                main_branches,
                amt_branches,
                mean_segment_length,
                max_segment_length,
                mean_thickness,
                max_thickness,
                sholl_depth,
                sholl_max,
                (radii, sholl_counts),
                sholl_auc,
                sholl_dec,
                soma_circularity,
                soma_feret_diam,
                branching_nodes,
                cha,
                density,
            ) = feats
            df = {}
            for r, c in zip(radii, sholl_counts):
                df[f"sholl_{int(r)}"] = c

            records.append(
                {
                    "slide": imgs[idx].stem,
                    "label": classes[idx],
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2,
                    "cx": cx,
                    "cy": cy,
                    "soma_area": soma_area,
                    "cell_area": cell_area,
                    "tortuosity": tortuosity,
                    "terminal_pts": terminal_pts,
                    "main_branches": main_branches,
                    "amt_branches": amt_branches,
                    "mean_segment_length": mean_segment_length,
                    "max_segment_length": max_segment_length,
                    "mean_thickness": mean_thickness,
                    "max_thickness": max_thickness,
                    "sholl_depth": sholl_depth,
                    "sholl_max": sholl_max,
                    "sholl_auc": sholl_auc,
                    "sholl_decay": sholl_dec,
                    "soma_circularity": soma_circularity,
                    "soma_feret_diameter": soma_feret_diam,
                    "branching_nodes": branching_nodes,
                    "cha": cha,
                    "density": density,
                    **df,
                }
            )

    print(f"{len(records)} cells saved after slide {imgs[idx].stem}")


df = pd.DataFrame(records)
out_path = Path("./results/new/microglia_features_raw.parquet")

out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(out_path, index=False)

print(f"Saved {len(df)} cells to {out_path}")
print(f"Skipped: {skips}")

  0%|          | 13/9356 [00:00<02:21, 66.24it/s]

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25


  0%|          | 38/9356 [00:00<01:31, 101.83it/s]

26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49


  1%|          | 61/9356 [00:00<01:33, 99.44it/s] 

...

  8%|▊         | 742/9356 [00:07<01:45, 81.98it/s]

729
730
731
732
733
734
735
736
737
738
739
740
741
742


  8%|▊         | 751/9356 [00:07<02:07, 67.25it/s]

743
744
745
746
747
748
749
750
751
752
753
754


  8%|▊         | 760/9356 [00:07<02:01, 70.88it/s]

## Find Slice Splits

In [ ]:
import numpy as np
import cv2
import pyvips
from pathlib import Path
from scipy import ndimage as ndi


def find_split_xs(overview, min_gap_px=20, smooth_sigma=25):
    gray = cv2.cvtColor(overview, cv2.COLOR_RGB2GRAY)
    mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((25, 25), np.uint8))

    profile = mask.mean(axis=0).astype(np.float32)
    profile_smooth = ndi.gaussian_filter1d(profile, sigma=smooth_sigma)

    low = profile_smooth < np.percentile(profile_smooth, 40)
    labels, n = ndi.label(low)

    split_xs = []
    for i in range(1, n + 1):
        cols = np.where(labels == i)[0]
        if len(cols) >= min_gap_px:
            split_xs.append(int((cols[0] + cols[-1]) / 2))

    return split_xs, profile_smooth, mask


def visualize_splits(overview, downsample=16, min_gap_px=20):
    split_xs, profile, mask = find_split_xs(overview, min_gap_px=min_gap_px)

    fig, ax = plt.subplots(1, 1, figsize=(16, 10))
    ax.imshow(overview)

    for x in split_xs:
        ax.axvline(x, color="cyan", linestyle="--", linewidth=2)

    ax.axis("off")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(16, 3))
    plt.plot(profile)
    for x in split_xs:
        plt.axvline(x, color="red", linestyle="--")
    plt.tight_layout()
    plt.show()

    return split_xs


def downsample_and_save(
    wsi_paths,
    output_dir,
    downsample=64,
    crop=None,
):

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    results = []

    for wsi_path in wsi_paths:
        wsi_path = Path(wsi_path)
        print(f"Processing: {wsi_path.name}")

        img = pyvips.Image.new_from_file(str(wsi_path), access="random")
        full_w, full_h = img.width, img.height

        if crop is not None:
            cx, cy, cw, ch = [int(v) for v in crop]
            img = img.crop(cx, cy, cw, ch)
            full_w, full_h = cw, ch

        overview = img.resize(1 / downsample)

        if overview.bands == 4:
            overview = overview.flatten()

        stem = wsi_path.stem
        out_path = output_dir / f"{stem}_ds{downsample}.png"

        overview.pngsave(str(out_path))

        thumb_w, thumb_h = overview.width, overview.height
        print(
            f"  {full_w}x{full_h} -> {thumb_w}x{thumb_h}  " f"(saved: {out_path.name})"
        )

        results.append(
            {
                "wsi_path": wsi_path,
                "out_path": out_path,
                "downsample": downsample,
                "full_w": full_w,
                "full_h": full_h,
                "thumb_w": thumb_w,
                "thumb_h": thumb_h,
            }
        )

    print(f"\nDone. {len(results)} image(s) saved to: {output_dir}")
    return results


def load_saved_overview(out_path):
    arr = cv2.imread(str(out_path))
    if arr is None:
        raise FileNotFoundError(f"Could not read: {out_path}")
    return cv2.cvtColor(arr, cv2.COLOR_BGR2RGB)


def scale_splits_to_fullres(split_xs_overview, downsample):
    return [int(x * downsample) for x in split_xs_overview]

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import random
import pyvips
import numpy as np
import cv2
from tqdm import tqdm
import pandas as pd
from scripts.filters import *
from scripts.utils import *
from cell_analysis.extract_features import *

IMAGE_DIR = "/mnt/d/microglia_data"
imgs = [f for f in Path(IMAGE_DIR).glob("./*.tiff")]
PARQUET_DIR = "../cell_localization/model_training/clustered_bboxes"
parquet = [f for f in Path(PARQUET_DIR).glob("./*.parquet")]
classes = list(map((lambda x: 0 if "EV" in x else 1), [f.name for f in imgs]))

indices = [0, 9, 2, 3, 4, 11, 10, 7, 8, 5, 6, 1]
parquets = [parquet[i] for i in indices]

print(imgs)
print(parquets)
print(classes)

In [ ]:
downsample_and_save(imgs, "./downsampled_images/")

In [ ]:
DOWNSAMPLE_DIR = "./downsampled_images/"
downsampled = [f for f in Path(DOWNSAMPLE_DIR).glob("./*.png")]
print(downsampled)

In [ ]:
final_splits = {}
for file in downsampled:
    overview = load_saved_overview(file)
    split_xs_thumb = visualize_splits(
        overview=overview,
        downsample=64,
        min_gap_px=30,
    )
    res = scale_splits_to_fullres(split_xs_thumb, downsample=64)
    final_splits[file.stem] = res
    print(f"{file.stem}: {res}")

## Split Slides by Slice

In [5]:
import numpy as np
import pandas as pd

slice_splits = {
    "TPO_60": [33280, 66240, 102208, 134080],
    "TPO_61_EV": [26880, 59648, 97664, 134464],
    "TPO_62_EV2": [18624, 59264, 98944, 140416],
    "TPO_62_EV": [23680, 52288, 82880, 115648],
    "TPO_63_TPO": [27264, 59584, 91328, 125952],
    "TPO_64_EV2": [38976, 84928, 121472, 159616],
    "TPO_64_EV": [28928, 60864, 97280, 137152],
    "TPO_65_TPO2": [35200, 88256, 140352],
    "TPO_65_TPO": [32256, 64768, 100672, 141120],
    "TPO_66_EV2": [30144, 62208, 92480, 121584],
    "TPO_66_EV": [27904, 74816, 124416],
    "TPO_67_TPO": [27712, 60352, 94336, 127872],
}
df = pd.read_parquet("./results/new/microglia_features_cleaned.parquet")


def assign_slice(x2, slide_name):
    splits = slice_splits.get(slide_name)
    if splits is None or len(splits) == 0 or pd.isna(x2):
        return np.nan
    return int(np.searchsorted(splits, x2, side="left") + 1)


df["brain_slice"] = df.apply(lambda row: assign_slice(row["x2"], row["slide"]), axis=1)

df.to_parquet("./results/new/microglia_features_slice.parquet", index=False)

# Second analysis 

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import random
import pyvips
import numpy as np
import cv2
from tqdm import tqdm
import pandas as pd
from scripts.filters import *
from scripts.utils import *
from cell_analysis.extract_features import *

IMAGE_DIR = "/mnt/d/microglia_data"
imgs = [f for f in Path(IMAGE_DIR).glob("./*.tiff")]
PARQUET_DIR = "../cell_localization/clustered_bboxes"
parquet = [f for f in Path(PARQUET_DIR).glob("./*.parquet")]
classes = list(map((lambda x: 0 if "EV" in x else 1), [f.name for f in imgs]))

indices = [0, 9, 2, 3, 4, 11, 10, 7, 8, 5, 6, 1]
parquets = [parquet[i] for i in indices]

print(imgs)
print(parquets)
print(classes)

[PosixPath('/mnt/d/microglia_data/TPO_60.tiff'), PosixPath('/mnt/d/microglia_data/TPO_61_EV.tiff'), PosixPath('/mnt/d/microglia_data/TPO_62_EV2.tiff'), PosixPath('/mnt/d/microglia_data/TPO_62_EV.tiff'), PosixPath('/mnt/d/microglia_data/TPO_63_TPO.tiff'), PosixPath('/mnt/d/microglia_data/TPO_64_EV2.tiff'), PosixPath('/mnt/d/microglia_data/TPO_64_EV.tiff'), PosixPath('/mnt/d/microglia_data/TPO_65_TPO2.tiff'), PosixPath('/mnt/d/microglia_data/TPO_65_TPO.tiff'), PosixPath('/mnt/d/microglia_data/TPO_66_EV2.tiff'), PosixPath('/mnt/d/microglia_data/TPO_66_EV.tiff'), PosixPath('/mnt/d/microglia_data/TPO_67_TPO.tiff')]
[PosixPath('../cell_localization/clustered_bboxes/TPO_60_bboxes.parquet'), PosixPath('../cell_localization/clustered_bboxes/TPO_61_EV_bboxes.parquet'), PosixPath('../cell_localization/clustered_bboxes/TPO_62_EV2_bboxes.parquet'), PosixPath('../cell_localization/clustered_bboxes/TPO_62_EV_bboxes.parquet'), PosixPath('../cell_localization/clustered_bboxes/TPO_63_TPO_bboxes.parquet'

In [2]:
from tqdm import tqdm
import random
from scipy.spatial import cKDTree


random.seed(25)
skips = 0
records = []

for idx, file in enumerate(parquets):
    df = pd.read_parquet(file)
    df["abs_cx"] = (df["x1"] + df["cx"]).astype(int)
    df["abs_cy"] = (df["y1"] + df["cy"]).astype(int)
    tree = cKDTree(df[["abs_cx", "abs_cy"]].to_numpy())

    bbs = extract_bbs_centroid(str(file), [])
    vips_img = pyvips.Image.new_from_file(imgs[idx], access="sequential")

    for bb in tqdm(bbs):

        x1, y1, x2, y2, cx, cy = (
            bb["x1"],
            bb["y1"],
            bb["x2"],
            bb["y2"],
            bb["cx"],
            bb["cy"],
        )
        cx_abs = x1 + cx
        cy_abs = y1 + cy
        crop = vips_img.crop(x1, y1, x2 - x1, y2 - y1)

        patch = np.ndarray(
            buffer=crop.write_to_memory(),
            dtype=np.uint8,
            shape=[crop.height, crop.width, crop.bands],
        )

        if patch.shape[-1] > 3:
            patch = patch[:, :, :3]
        try:
            feats = extract_feats2(patch, (cx, cy), tree, df, (x1, y1))
        except Exception as e:
            skips += 1
            print(f"Exception: {e}")
            continue

        if feats is None:
            print("feats none")
            skips += 1
            continue

        radii, counts = feats["SHOLL_ANALYSIS"]
        sholl_df = {f"sholl_{int(r)}": c for r, c in zip(radii, counts)}

        records.append(
            {
                "slide": imgs[idx].stem,
                "label": classes[idx],
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "cx": cx,
                "cy": cy,
                "soma_area": feats["SOMA_AREA"],
                "soma_radius": feats["SOMA_RADIUS"],
                "cell_area": feats["CELL_AREA"],
                "solidity": feats["SOLIDITY"],
                "circularity": feats["CIRCULARITY"],
                "n_branches": feats["N_BRANCHES"],
                "n_junctions": feats["N_JUNCTIONS"],
                "mean_branch_len": feats["MEAN_BRANCH_LENGTH"],
                "skel_length": feats["SKEL_LENGTH"],
                "cha": feats["CHA"],
                "terminal_pts": feats["TERMINAL_PTS"],
                **sholl_df,
            }
        )

    print(f"{len(records)} cells saved after slide {imgs[idx].stem}")


df = pd.DataFrame(records)
out_path = Path("./results/new/microglia_features_raw.parquet")

out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(out_path, index=False)

print(f"Saved {len(df)} cells to {out_path}")
print(f"Skipped: {skips}")

  1%|          | 60/9356 [00:11<41:31,  3.73it/s] 

Exception: index pointer size 0 should be 1


  3%|▎         | 304/9356 [01:01<30:19,  4.98it/s]

Exception: index pointer size 0 should be 1


  3%|▎         | 317/9356 [01:04<30:03,  5.01it/s]

...

Exception: index pointer size 0 should be 1


100%|█████████▉| 9138/9159 [29:36<00:03,  6.24it/s]

Exception: min() arg is an empty sequence


100%|██████████| 9159/9159 [29:39<00:00,  5.15it/s]


112930 cells saved after slide TPO_67_TPO
Saved 112930 cells to results/new/microglia_features_raw.parquet
Skipped: 1568


## Clean DF

In [4]:
import re
import pandas as pd

df = pd.read_parquet("./results/new/microglia_features_raw.parquet")

for n in range(10, 251, 10):
    good = f"sholl_{n}"
    bad = f"sholl_{n-1}"

    if good in df.columns and bad in df.columns:
        df[good] = df[good].combine_first(df[bad])

bad_cols = [f"sholl_{n}" for n in range(9, 250, 10)]
df = df.drop(columns=[c for c in bad_cols if c in df.columns])
non_sholl = [c for c in df.columns if not c.startswith("sholl_")]
sholl_sorted = sorted(
    [c for c in df.columns if c.startswith("sholl_")],
    key=lambda x: int(re.search(r"\d+", x).group()),
)
df = df[non_sholl + sholl_sorted]
df["sholl_250"] = df["sholl_250"].fillna(0)
df
df.to_parquet("./results/new/microglia_features_cleaned.parquet")